# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [ ]:
import torch
from dataset import get_dataloader

loader = get_dataloader("./data/uco3d_data_raw", batch_size=1, num_samples=1)


In [ ]:
import torch.nn as nn
from dataset import GaussianSplats

class POC_Network(nn.Module):
    def __init__(self, dim):
        super(POC_Network, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        x = self.model(x)
        return x
    
    def reconstruct_splats(self, splats):
        """Convert splats to flat tensor, apply network, and reconstruct splats"""
        # Flatten all splat parameters into a single vector per gaussian
        means = splats.means  # (N, 3)
        sh0 = splats.sh0     # (N, 3)
        scales = splats.scales  # (N, 3)
        quats = splats.quats    # (N, 4)
        opacities = splats.opacities  # (N,)
        
        # Concatenate: 3 + 3 + 3 + 4 + 1 = 14 dims per gaussian
        flat = torch.cat([means, sh0, scales, quats, opacities.unsqueeze(-1)], dim=-1)  # (N, 14)
        
        # Apply network
        reconstructed_flat = self.model(flat)  # (N, 14)
        
        # Split back into components
        means_recon = reconstructed_flat[:, 0:3]
        sh0_recon = reconstructed_flat[:, 3:6]
        scales_recon = reconstructed_flat[:, 6:9]
        quats_recon = reconstructed_flat[:, 9:13]
        opacities_recon = reconstructed_flat[:, 13:14].squeeze(-1)
        
        # Create reconstructed splats object
        recon_splats = GaussianSplats(
            means=means_recon,
            sh0=sh0_recon,
            scales=scales_recon,
            quats=quats_recon,
            opacities=opacities_recon,
            shN=splats.shN,
            fg_mask=splats.fg_mask,
        )
        return recon_splats

network = POC_Network(dim=14)
